# 🧠 Asistente Experto en Micron Technology
## Proyecto Final — IA Generativa (RAG + Gemini + LangGraph)

**Dominio elegido:** finanzas corporativas y gobierno corporativo de **Micron Technology, Inc. (NASDAQ: MU)**.

Este notebook construye un agente de IA que responde preguntas sobre Micron a partir de su
propia base de conocimiento vectorial, construida con documentos públicos presentados ante la
SEC (Securities and Exchange Commission de EE. UU.):

| Documento | Tipo | Contenido |
|---|---|---|
| `micron_10K_FY2025` | Form 10-K (informe anual) | Negocio, factores de riesgo, resultados financieros FY2025 |
| `micron_10K_FY2024` | Form 10-K (informe anual) | Negocio, factores de riesgo, resultados financieros FY2024 |
| `micron_proxy_2025` | DEF 14A (proxy statement) | Gobierno corporativo, compensación ejecutiva, junta directiva (nov. 2025) |
| `micron_proxy_2024` | DEF 14A (proxy statement) | Gobierno corporativo, compensación ejecutiva, junta directiva (nov. 2024) |

Estos 4 documentos superan ampliamente el mínimo exigido (3 documentos / ~20 páginas) y cubren
dos dimensiones complementarias del dominio: **desempeño financiero** (10-K) y **gobierno
corporativo / compensación** (proxy statement), a lo largo de dos años fiscales consecutivos —
lo que permite además preguntas comparativas interanuales.

**Stack tecnológico (según guideline del proyecto):**
- LLM y Embeddings: **Google Gemini** (`langchain-google-genai`)
- Base de conocimiento vectorial: **ChromaDB** (`langchain-chroma`)
- Framework de agente: **LangGraph**
- Memoria de conversación: `langgraph.checkpoint.memory.InMemorySaver`
- Entorno: Jupyter Notebook

> ℹ️ **Nota sobre la elección del LLM:** se evaluó usar la API de Claude (Anthropic) en lugar de
> Gemini. Se descartó porque (1) el guideline del proyecto exige explícitamente Gemini para LLM
> *y* embeddings, y la rúbrica evalúa "Chroma + Gemini Embeddings" y "RAG con Gemini" como
> criterios propios, y (2) Anthropic no ofrece una API pública de embeddings, por lo que aun
> usando Claude como LLM habría sido necesario un proveedor de embeddings distinto. Se mantiene
> Gemini de extremo a extremo para cumplir el stack exigido sin riesgo en la evaluación.


## 📦 1. Instalación de dependencias

In [50]:
# Ejecutar una sola vez (descomentar si faltan paquetes)
%pip install -q langchain langchain-google-genai langchain-classic langchain-community \
    langchain-chroma langchain-text-splitters langgraph chromadb pypdf beautifulsoup4 \
    lxml python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\imsmo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 🔑 2. Configuración de la API key de Gemini

Guarda tu clave en un archivo `.env` (mismo directorio que este notebook) con el contenido:

```
GEMINI_API_KEY=tu_api_key_aqui
```

**Nunca** subas el `.env` a un repositorio ni escribas la clave directamente en el código
(ver `.env.example` incluido en el proyecto).


In [51]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "No se encontró GEMINI_API_KEY. Crea un archivo .env con GEMINI_API_KEY=tu_api_key "
        "(ver .env.example) antes de continuar."
    )

# Modelos de Gemini utilizados en el proyecto
CHAT_MODEL = "gemini-2.5-flash"#"gemini-2.5-flash-lite"         # LLM para generación de respuestas
EMBEDDING_MODEL = "gemini-embedding-001"  # Modelo de embeddings de Gemini (sin prefijo "models/" en el SDK actual)

print("✅ API key cargada correctamente.")


✅ API key cargada correctamente.


## 📚 3.— Base de conocimiento: carga de documentos

Coloca los 4 documentos de Micron dentro de `data/micron_docs/` (junto a este notebook).
El loader admite tanto **PDF** como **HTML** (tal como se descargan directamente de SEC EDGAR),
para que no sea obligatorio convertirlos.

Fuentes oficiales (SEC EDGAR) usadas para este proyecto:

- 10-K FY2025: https://www.sec.gov/Archives/edgar/data/723125/000072312525000028/mu-20250828.htm
- 10-K FY2024: https://www.sec.gov/Archives/edgar/data/723125/000072312524000027/mu-20240829.htm
- Proxy 2025 (DEF 14A): https://www.sec.gov/Archives/edgar/data/723125/000072312525000038/mu-20251125.htm
- Proxy 2024 (DEF 14A): https://www.sec.gov/Archives/edgar/data/723125/000072312524000039/mu-20241126.htm


In [52]:
import re
import glob
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader

DATA_DIR = Path("data/micron_docs")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def inferir_metadatos(nombre_archivo: str) -> dict:
    """Detecta tipo de documento y año fiscal a partir del NOMBRE del archivo, sin
    depender de una convención de nombres exacta (funciona tanto si renombraste los PDFs
    como si conservaste el nombre original de SEC EDGAR, p. ej.
    '2024 Definitive Proxy_Final_11.26.24.pdf' o 'mu-20250828.pdf')."""
    nombre_lower = nombre_archivo.lower()
    if "10-k" in nombre_lower or "10k" in nombre_lower:
        tipo = "10-K"
    elif "proxy" in nombre_lower or "def 14a" in nombre_lower or "def14a" in nombre_lower:
        tipo = "Proxy Statement (DEF 14A)"
    else:
        tipo = "Documento Micron"

    anios = re.findall(r"(20\d{2})", nombre_archivo)
    anio_fiscal = int(anios[0]) if anios else None

    return {"tipo": tipo, "anio_fiscal": anio_fiscal}


def limpiar_texto_pdf(texto: str) -> str:
    """Elimina ruido típico de la extracción de PDFs de SEC EDGAR (Paso 2 del guideline:
    'eliminación de ruido'):

    1. Texto con letras separadas por espacios, frecuente en casillas de formularios y
       portadas (ej. 'F i l e d  b y' -> 'Filed by'). No afecta al texto normal porque el
       patrón solo coincide cuando CADA carácter individual está seguido de un espacio.
    2. Espacios y saltos de línea redundantes.
    """
    texto = re.sub(r"\b(?:\w ){1,}\w\b", lambda m: m.group(0).replace(" ", ""), texto)
    texto = re.sub(r"[ \t]{2,}", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


documentos = []
archivos = sorted(glob.glob(str(DATA_DIR / "*.pdf"))) + \
           sorted(glob.glob(str(DATA_DIR / "*.htm"))) + \
           sorted(glob.glob(str(DATA_DIR / "*.html")))

if not archivos:
    print(f"⚠️  No se encontraron documentos en {DATA_DIR.resolve()}.")
    print("   Descarga los 4 documentos de Micron (ver celda anterior) y colócalos en esa carpeta.")

for ruta in archivos:
    path = Path(ruta)
    stem = path.stem
    if path.suffix.lower() == ".pdf":
        loader = PyPDFLoader(str(path))
    else:
        loader = BSHTMLLoader(str(path), open_encoding="utf-8")

    docs = loader.load()

    # Limpieza de ruido + metadatos (tipo de documento y año fiscal) para citar mejor las fuentes
    extra = inferir_metadatos(path.name)
    for d in docs:
        d.page_content = limpiar_texto_pdf(d.page_content)
        d.metadata["source_name"] = stem
        d.metadata.update(extra)

    documentos.extend(docs)
    print(f"  - {path.name}: {len(docs)} página(s)/sección(es) cargadas ({extra['tipo']}, FY{extra['anio_fiscal']})")

print(f"\n✅ Total: {len(documentos)} fragmentos cargados desde {len(archivos)} documento(s).")


  - 2024 Proxy Statement.pdf: 138 página(s)/sección(es) cargadas (Proxy Statement (DEF 14A), FY2024)
  - 2025 Proxy Statement.pdf: 112 página(s)/sección(es) cargadas (Proxy Statement (DEF 14A), FY2025)
  - micron_10K_FY2024.pdf: 117 página(s)/sección(es) cargadas (10-K, FY2024)
  - micron_10K_FY2025.pdf: 110 página(s)/sección(es) cargadas (10-K, FY2025)

✅ Total: 477 fragmentos cargados desde 4 documento(s).


## ✂️ 4.— Preprocesamiento y chunking

Se aplica `RecursiveCharacterTextSplitter` para segmentar los documentos en fragmentos manejables,
con solapamiento para no perder contexto entre secciones (relevante para tablas financieras que
continúan de una página a otra).

> 📎 Con ~450-480 páginas totales, un chunk_size de 1500 caracteres genera entre ~1000 y ~1300
> chunks — es la proporción esperada (≈2.5-3 chunks/página) y un volumen perfectamente manejable
> para la API de embeddings de Gemini. **No hace falta agrandar el chunk_size**; lo que sí merece
> la pena es limpiar el ruido de extracción (celda anterior) antes de trocear, porque un chunk
> "sucio" (ej. la portada con casillas ☒/☐ del proxy) aporta poco valor de recuperación
> independientemente de su tamaño.


In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=250,
    separators=["\n\n", "\n", ". ", " ", ""],
)

docs_split = text_splitter.split_documents(documentos)

# Descarta chunks casi vacíos o puramente boilerplate (portadas, casillas de formulario, etc.)
# que aportan poco al retriever y solo añaden ruido/coste de embeddings.
docs_split = [d for d in docs_split if len(d.page_content.strip()) >= 200]

print(f"Se crearon {len(docs_split)} chunks de texto a partir de {len(documentos)} páginas/secciones.")
print("\nEjemplo de chunk:\n")
print(docs_split[0].page_content[:500])
print("\nMetadatos:", docs_split[0].metadata)


Se crearon 1263 chunks de texto a partir de 477 páginas/secciones.

Ejemplo de chunk:

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
SCHEDULE 14A
Proxy Statement Pursuant to Section 14(a) of
the Securities Exchange Act of 1934 (Amendment No. )
Filed by the Registrant ☒
Filed by a party other than the Registrant ☐
Check the appropriate box:
☐ Preliminary Proxy Statement
☐ Confidential, for Use of the Commission Only (as permitted by Rule 14a-6(e)(2))
☒ Definitive Proxy Statement
☐ Definitive Additional Materials
☐ Soliciting Material under §240.14a-12
Micr

Metadatos: {'producer': 'Wdesk Fidelity Content Translations Version 010.005.115', 'creator': 'Workiva', 'creationdate': '2024-11-26T22:03:11+00:00', 'moddate': '2024-11-26T22:03:11+00:00', 'title': '2024 Definitive Proxy', 'author': 'anonymous', 'source': 'data\\micron_docs\\2024 Proxy Statement.pdf', 'total_pages': 138, 'page': 0, 'page_label': '1', 'source_name': '2024 Proxy Statement', 'tipo': 'Proxy Sta

## 🧬 5.— Base de conocimiento vectorial (ChromaDB + Gemini Embeddings)

Se generan los embeddings con **Gemini** (`GoogleGenerativeAIEmbeddings`) y se indexan en una
colección persistente de **ChromaDB**.

> ⏱️ Esta celda puede tardar varios minutos según el número de chunks y los límites de la API
> gratuita de Gemini. Si tienes muchos documentos, puedes reducir el alcance quitando temporalmente
> uno de los 10-K para probar el pipeline antes de escalar (recomendación del guideline).
>
> ⚠️ **Límite de la API gratuita:** el free tier de Gemini permite un máximo de **100 peticiones
> de embeddings por minuto** y **1000 peticiones al día** por modelo. Con varios cientos/miles de
> chunks es fácil superar cualquiera de los dos y recibir un error `429 RESOURCE_EXHAUSTED`. Por
> eso esta celda:
>
> 1. Indexa en **lotes pequeños con pausas entre ellos** (para el límite por minuto).
> 2. **Reintenta automáticamente** (con espera creciente) si la API devuelve ese error.
> 3. **Reanuda desde donde se quedó** si ya hay chunks indexados de una ejecución anterior (por
>    ejemplo, si ayer se cortó por el límite diario) — no vuelve a gastar cuota re-embebiendo lo
>    que ya está guardado en `chroma_db_micron`.
>
> ⚠️ Importante para que la reanudación funcione bien: no cambies los archivos de
> `data/micron_docs/` ni los parámetros de chunking entre una ejecución y la siguiente, o el
> número de chunks ya indexados dejará de corresponder a los mismos documentos.


In [54]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=API_KEY,
)

PERSIST_DIR = "./chroma_db_micron"
COLLECTION_NAME = "micron_knowledge_base"

# Vectorstore al que iremos añadiendo los chunks en lotes (en vez de Chroma.from_documents,
# que intenta embeber todo de una vez y es lo que dispara el 429 con el free tier de Gemini).
# Si ya existe la colección (de una ejecución anterior), se reutiliza tal cual.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

BATCH_SIZE = 20      # nº de chunks por petición a la API de embeddings
PAUSE_SECONDS = 15   # pausa entre lotes para no superar las 100 peticiones/minuto del free tier
MAX_REINTENTOS = 5

total = len(docs_split)

# --- Reanudación ---
# Si la colección ya tiene chunks (por ejemplo, porque ayer se cortó a mitad por el límite
# diario de la API), retomamos desde ahí en vez de volver a embeber los primeros N chunks.
ya_indexados = vectorstore._collection.count()
if ya_indexados > 0:
    print(f"↩️  La colección ya tiene {ya_indexados} chunks indexados de una ejecución anterior.")
    print(f"   Retomando desde el chunk {ya_indexados} (de {total} en total).\n")

if ya_indexados >= total:
    print("✅ Todos los chunks ya estaban indexados. Nada que hacer.")
else:
    inicio = ya_indexados
    num_lotes = (total - 1) // BATCH_SIZE + 1

    for idx, i in enumerate(range(inicio, total, BATCH_SIZE), start=inicio // BATCH_SIZE + 1):
        lote = docs_split[i : i + BATCH_SIZE]
        intento = 0
        while True:
            try:
                vectorstore.add_documents(lote)
                break
            except Exception as e:
                # Reintentamos tanto en límites de cuota (429 / RESOURCE_EXHAUSTED) como en
                # caídas puntuales del servicio de Google (503 / UNAVAILABLE, 500, 504), que
                # son errores transitorios y no tienen relación con nuestra cuota diaria.
                es_reintentable = any(
                    codigo in str(e)
                    for codigo in ["RESOURCE_EXHAUSTED", "429", "UNAVAILABLE", "503", "500", "504"]
                )
                if es_reintentable and intento < MAX_REINTENTOS:
                    intento += 1
                    espera = 20 * intento
                    print(f"  ⏳ Error temporal de la API en el lote {idx}/{num_lotes} ({e.__class__.__name__}). "
                          f"Reintento {intento}/{MAX_REINTENTOS} en {espera}s...")
                    time.sleep(espera)
                else:
                    raise
        print(f"  ✅ Lote {idx}/{num_lotes} indexado ({min(i + BATCH_SIZE, total)}/{total} chunks)")
        time.sleep(PAUSE_SECONDS)

    print(f"\n✅ Base de conocimiento vectorial completa: {total} chunks en '{PERSIST_DIR}'.")


↩️  La colección ya tiene 1263 chunks indexados de una ejecución anterior.
   Retomando desde el chunk 1263 (de 1263 en total).

✅ Todos los chunks ya estaban indexados. Nada que hacer.


## ✅ 6.— Verificar que la colección responde antes de conectar el agente

Se lanzan consultas de prueba directamente contra Chroma (sin pasar por el LLM) para confirmar
que la recuperación semántica funciona correctamente antes de construir el agente.


In [55]:
consultas_prueba = [
    "¿Cuáles fueron los ingresos totales de Micron en el año fiscal 2025?",
    "¿Quién es el CEO de Micron y cuál es su compensación?",
    "¿Cuáles son los principales riesgos relacionados con China?",
]

for consulta in consultas_prueba:
    print(f"\n{'='*70}\n🔎 Consulta: {consulta}\n{'='*70}")
    resultados = vectorstore.similarity_search(consulta, k=2)
    for i, doc in enumerate(resultados, 1):
        fuente = doc.metadata.get("source_name", "desconocida")
        pagina = doc.metadata.get("page", "?")
        print(f"\n[{i}] Fuente: {fuente} (página {pagina})")
        print(doc.page_content[:400], "...")



🔎 Consulta: ¿Cuáles fueron los ingresos totales de Micron en el año fiscal 2025?

[1] Fuente: micron_10K_FY2025 (página 63)
Micron Technology, Inc.
Consolidated Statements of Operations
(In millions, except per share amounts)
For the year ended
August 28,
2025
August 29,
2024
August 31,
2023
Revenue $ 37,378 $ 25,111 $ 15,540 
Cost of goods sold 22,505 19,498 16,956 
Gross margin 14,873 5,613 (1,416) 
Research and development 3,798 3,430 3,114 
Selling, general, and administrative 1,205 1,129 920 
Restructure and asset ...

[2] Fuente: micron_10K_FY2025 (página 51)
Results of Operations
Consolidated Results
For the year ended 2025 2024 2023
Revenue $ 37,378 100 % $ 25,111 100 % $ 15,540 100 %
Cost of goods sold 22,505 60 % 19,498 78 % 16,956 109 %
Gross margin 14,873 40 % 5,613 22 % (1,416) (9) %
Research and development 3,798 10 % 3,430 14 % 3,114 20 %
Selling, general, and administrative 1,205 3 % 1,129 4 % 920 6 %
Restructure and asset impairments 39 — %  ...

🔎 Consulta: ¿Quién es

## 🎯 7.— Diseño del System Prompt (justificado)

**Rol:** analista experto en los informes públicos de Micron Technology (10-K y proxy statements).

**Tono:** profesional, preciso, basado en evidencia — como un analista financiero que cita sus fuentes.

**Limitaciones definidas explícitamente:**
1. Responde **únicamente** con información recuperada de los documentos indexados; no debe
   inventar cifras ni completar huecos con conocimiento general del modelo.
2. Cuando no encuentra información suficiente en la base de conocimiento, debe decirlo
   explícitamente en lugar de especular ("No tengo información suficiente en los documentos
   indexados para responder con precisión").
3. **No** proporciona recomendaciones de inversión ni juicios de valor sobre si comprar/vender
   la acción — se limita a reportar lo que dicen los documentos. Esto es importante porque el
   agente maneja información financiera pública y no es un asesor financiero regulado.
4. Cita el documento y, cuando sea posible, el año fiscal de origen de cada dato relevante, para
   dar trazabilidad a la respuesta (importante en un dominio donde la precisión y la fuente
   importan).
5. Mantiene coherencia con el historial de la conversación (memoria) para resolver preguntas de
   seguimiento sin que el usuario tenga que repetir contexto.

**Justificación de estas decisiones:** el dominio (informes financieros y de gobierno corporativo)
es sensible a errores de precisión — una cifra incorrecta o una alucinación puede ser
tomada como un dato financiero real. Por eso el prompt prioriza la fidelidad a la fuente sobre la
fluidez, y prohíbe explícitamente la asesoría de inversión (evita responsabilidad legal y se ciñe
al alcance del proyecto: informar, no aconsejar).


In [56]:
SYSTEM_PROMPT = """Eres un analista experto en los informes públicos de Micron Technology, Inc.
(NASDAQ: MU) presentados ante la SEC: sus informes anuales (Form 10-K) y sus proxy statements
(DEF 14A) de los últimos dos años fiscales.

INSTRUCCIONES:
1. Usa SIEMPRE la herramienta `buscar_en_informes_micron` para fundamentar tus respuestas antes
   de responder preguntas sobre Micron (finanzas, riesgos, negocio, gobierno corporativo,
   compensación de ejecutivos, junta directiva, etc.).
2. Responde ÚNICAMENTE con base en la información recuperada de los documentos indexados. No
   inventes cifras ni completes huecos con conocimiento general.
3. Si la información recuperada no es suficiente para responder con precisión, dilo
   explícitamente: "No tengo información suficiente en los documentos indexados para responder
   con precisión a esa pregunta."
4. Cuando cites datos, menciona el documento y año fiscal de origen (por ejemplo: "según el 10-K
   de FY2025...").
5. NO des recomendaciones de inversión ni opiniones sobre si comprar, vender o mantener la
   acción de Micron. Si te lo piden, aclara que solo puedes reportar lo que indican los
   documentos públicos, no ofrecer asesoría financiera.
6. Mantén coherencia con las preguntas anteriores de la conversación.
7. Si la pregunta no tiene relación con Micron o sus informes, indícalo brevemente y redirige
   al usuario al alcance del asistente.
8. Responde siempre en español, de forma clara y profesional.
"""

## 🛠️ 8. Herramienta de recuperación (RAG tool)

In [57]:
from langchain.tools import tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

@tool
def buscar_en_informes_micron(query: str) -> str:
    """Busca información relevante en los informes 10-K y proxy statements (DEF 14A) de Micron
    Technology. Úsala para cualquier pregunta sobre finanzas, riesgos, negocio, segmentos,
    gobierno corporativo, junta directiva o compensación de ejecutivos de Micron."""
    docs = retriever.invoke(query)
    if not docs:
        return "No se encontró información relevante en los documentos indexados."

    resultados = []
    for doc in docs:
        fuente = doc.metadata.get("source_name", "desconocida")
        tipo = doc.metadata.get("tipo", "")
        anio = doc.metadata.get("anio_fiscal", "")
        pagina = doc.metadata.get("page", "?")
        resultados.append(
            f"[Fuente: {fuente} | {tipo} FY{anio} | página {pagina}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(resultados)

# Prueba directa de la herramienta
print(buscar_en_informes_micron.invoke({"query": "riesgos de concentración de clientes"})[:800])


[Fuente: micron_10K_FY2025 | 10-K FY2025 | página 31]
A significant portion of our revenue is concentrated with certain customers and end markets.
In 2025, over half of our total revenue came from our top ten customers. Among our end markets, approximately 
one-half of our total revenue was concentrated in the data center end market. A disruption in our relationship with 
any of our top customers or a significant decrease in demand for our data center products, or in the overall data 
center end market, could adversely affect our business. We could experience fluctuations in our customer base or 
the mix of revenue by customer or end market, as markets and strategies evolve. Demand for our products may 
fluctuate due to factors beyond our control. Our inability to qualify our products to m


## 🤖 9.— Agente en LangGraph (RAG + Gemini + memoria)

Se crea el agente con `create_agent` (LangGraph/LangChain), usando Gemini como LLM, la
herramienta de recuperación definida arriba, el system prompt justificado y un
`InMemorySaver` como checkpointer para dar memoria de conversación (Paso 5 del guideline).


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    temperature=0.2,
    google_api_key=API_KEY,
)

checkpointer = InMemorySaver()

agente_micron = create_agent(
    model=llm,
    tools=[buscar_en_informes_micron],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

print("Agente RAG de Micron creado correctamente (con memoria de conversación).")


✅ Agente RAG de Micron creado correctamente (con memoria de conversación).


## 💬 10. Función auxiliar para conversar con el agente

In [59]:
from langchain_core.messages import HumanMessage

def preguntar(pregunta: str, thread_id: str = "demo", mostrar_proceso: bool = False) -> str:
    """Envía una pregunta al agente dentro de un thread (conversación) concreto y devuelve la
    respuesta final. Si mostrar_proceso=True, imprime también qué herramientas usó el agente."""
    config = {"configurable": {"thread_id": thread_id}}
    respuesta = agente_micron.invoke(
        {"messages": [HumanMessage(content=pregunta)]},
        config=config,
    )

    if mostrar_proceso:
        for msg in respuesta["messages"]:
            tipo = msg.__class__.__name__
            if tipo == "AIMessage" and getattr(msg, "tool_calls", None):
                for tc in msg.tool_calls:
                    print(f"  🔧 Herramienta usada: {tc['name']} | Query: {tc['args'].get('query', '')}")

    return respuesta["messages"][-1].content


## 🧪 11.— Interacción en el notebook: 5+ preguntas documentadas

A continuación, un mínimo de 5 preguntas de ejemplo (todas dentro del mismo `thread_id`, para que
el agente mantenga contexto). La **pregunta 5 depende explícitamente de la respuesta a la
pregunta 1**, demostrando que la memoria de conversación funciona (Paso 5 del guideline).

> Ejecuta esta celda tras rellenar `data/micron_docs/` y crear tu `.env` con tu propia
> `GEMINI_API_KEY`. Las respuestas se imprimen en vivo — documenta aquí los resultados reales
> obtenidos en tu ejecución para la entrega.


In [60]:
THREAD = "sesion-demo-1"

preguntas_ejemplo = [
    "¿Cuáles fueron los ingresos totales de Micron en el año fiscal 2025 y cómo se compararon "
    "con el año fiscal 2024?",
    "¿Cuáles son los principales riesgos que Micron identifica en relación con la concentración "
    "de clientes y con China?",
    "¿Quién es el CEO de Micron y desde cuándo ocupa el cargo?",
    "Según el proxy statement, ¿cómo se estructura la compensación variable de los ejecutivos?",
    "Teniendo en cuenta lo que me dijiste antes sobre los ingresos por año fiscal, "
    "¿cuál de los dos años fue mejor para Micron y por qué?",  # depende de la respuesta 1 (memoria)
]

for i, pregunta in enumerate(preguntas_ejemplo, 1):
    print(f"\n{'#'*70}\nPregunta {i}: {pregunta}\n{'#'*70}")
    respuesta = preguntar(pregunta, thread_id=THREAD, mostrar_proceso=True)
    print(f"\n🤖 Respuesta:\n{respuesta}")



######################################################################
Pregunta 1: ¿Cuáles fueron los ingresos totales de Micron en el año fiscal 2025 y cómo se compararon con el año fiscal 2024?
######################################################################
  🔧 Herramienta usada: buscar_en_informes_micron | Query: ingresos totales Micron año fiscal 2024 y 2025

🤖 Respuesta:
[{'type': 'text', 'text': 'Según el 10-K de FY2025 de Micron, los ingresos totales de la compañía fueron de $37,378 millones en el año fiscal 2025. Esto representa un aumento del 49% en comparación con el año fiscal 2024, cuando los ingresos totales fueron de $25,111 millones. El incremento se debió principalmente a un aumento en las ventas de productos DRAM y NAND.', 'extras': {'signature': 'CpoEARFNMg8QuL73piNT2Tx4M97Qurofhp7anierddAv4Im/EtvtDhDxvw23jqggtgwB05nTbbZAdoAxEgEyJbX3J3/aBlDybvf3PxWIHFWH2ezZANKqSaGXBjL+Q4w7Ny8MrIiZIDBB4Ue7eIgRkcFpVw4qRHlTX85CaBKhZO7M7w9PDZ6TbmZmgvMvBe20gBvwgsemAxkiAVTCuidNg+qEs

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 20.976708474s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

### 🔁 Ejemplo adicional — límite del agente (declina dar asesoría de inversión)

Esta pregunta no debería usar la herramienta de recuperación de forma útil para dar un consejo,
sino que debe activar la limitación definida en el system prompt (punto 5).


In [ ]:
respuesta_limite = preguntar(
    "¿Debería comprar acciones de Micron ahora mismo?",
    thread_id=THREAD,
)
print(respuesta_limite)


[{'type': 'text', 'text': 'Como analista de los informes públicos de Micron, mi función es proporcionarte información basada en los documentos indexados (informes 10-K y proxy statements DEF 14A). No estoy autorizado para dar recomendaciones de inversión ni opiniones sobre si comprar, vender o mantener acciones de Micron. Solo puedo reportar lo que indican los documentos públicos, no ofrecer asesoría financiera.', 'extras': {'signature': 'CtoDARFNMg/fTfU3N71wSfn6VWcTqB5lsGrM4fA+DNHgJ/tdYidDnIR+fVI3KjcgoSoC/ljaSSX1al9u9G79Ct204B+eX3N+DR2i6e18dEQ4+dxc66A4iuH1258z0qRMLFMpsgIE2/03YdEzigAnpbTWCHD/1vYM5f89+XbXgR4m+tZdEsJivnJYUGmgv7fQ4kTEr/5ej4tuYh9EqSKB9ApqNl0lPyhUnsuuJM5QMF5Xms0Qn4/YcZJtHn/p04EUou7u5DD27uOCZ1mtfmhY4+7JFQsa7VbX2TBhfoNsEfX0KctmA8QOorg6YEsC8SaSL7XO8XVMAij1G0H+SyNz6Rgwid2z8YPMf6PvNeaMXi3H+UF3/Vh6XzJGcb3WZAyL+dYRinm35boR7ccN9V+DdimvLgnvq4Y1ChLM8KUgBhhtH5ljHomzKsEiV1LC4V8KUY0Z6PXdrS/BPYSTCwAnNyXBCgF1JrnxWcDsMrdDQZQSBFUtOy36ZvM1cDMFWRwzGv663dsg5fo5IfRFwdumdemdDFg/0WNvXjUCY7Rw+EPRT

## 🧵 12. Verificación explícita de memoria multi-thread

Se demuestra que dos conversaciones (`thread_id` distintos) son independientes, y que dentro de
un mismo thread el agente sí recuerda lo dicho anteriormente.


In [ ]:
# Thread A: pregunta inicial
print(preguntar("¿Cuántos empleados tenía Micron aproximadamente en el último año fiscal reportado?", thread_id="thread-A"))

# Thread B: conversación totalmente distinta, no debería mezclar contexto con el thread A
print("\n---\n")
print(preguntar("Resume en una frase a qué se dedica Micron.", thread_id="thread-B"))

# Volvemos al thread A y hacemos una pregunta de seguimiento SIN repetir contexto
print("\n---\n")
print(preguntar("¿Y ese número cómo se compara con el año fiscal anterior?", thread_id="thread-A"))


[{'type': 'text', 'text': 'No tengo información suficiente en los documentos indexados para responder con precisión a esa pregunta. Los informes mencionan aspectos relacionados con los empleados, como programas de compensación y grupos de recursos, pero no especifican el número total de empleados para el último año fiscal reportado.', 'extras': {'signature': 'CusDARFNMg9Z2DjzkJxeP2mlyQhl3URgVPgh5jbCWmy3QtClU/D09992sLGtWZv3jGdtHKGbBcZKhjahlUdXk5tHqloD/ZFIIGXYEWOSU5xq0Hza3DL/xzDJnxy10em4EJYztYH4w6/0krKGMiCBavKeNow/zh0L7MyGIj952q/KnEMz5rU6vb5AOOEoqLIZcdq/Gz1+pc2RRlZUhdrGNerg20TyxoUl2BgFfd1uTS2UVai6YKVR5gO8Nze/DYdNditQ4Yo5Ia3gAXgFcyYqri1u77u+vHsxvIQtkAXpt0/7PgN5t70eHxQfqeIxosi1F6AdqxSVEUtZghI1YiFwLoKNVFMs3LXBp7izncK/b2vwsaH4wQnYyr8DE4jxz5K38LOg3NERJzOht/k/wtgLG2qGIShRbwjn8VW9EdUGpwtMj67V6orcDhYSsSk5zgxZed8wJHbI/yeEdtuW/vcEMZzVDBtmISyjqVnevcrXg7Z7nsAy6GE/uP6QPCyq7PU1jck1ky6Kh3y6+ytMNxkPlFXkfGYat194u3e1Gvf51vYAvQQZ6hEHkSaOEdmEUZu+W6q2AEpIJ2uI+LV6VareLLy9fffx3p6MmMTsgqFRgWOZHzP46q9BL1o27GO4iO

## 🗨️ 13. Extensión opcional (bonus) — Interfaz de chat con Streamlit

En vez de una celda de chat por `input()` dentro del notebook, este proyecto incluye una
interfaz visual de chat construida con **Streamlit** (`streamlit_app.py`, en la raíz del
proyecto), que reutiliza la misma base de conocimiento vectorial (ChromaDB) y el mismo agente
LangGraph + Gemini definidos arriba (mismo system prompt, misma herramienta de recuperación).

Para lanzarla (con la colección ya indexada, es decir, tras ejecutar la sección 5 al menos
una vez):

```bash
streamlit run streamlit_app.py
```

Se abre en el navegador en `http://localhost:8501`. Cada pestaña/sesión obtiene su propio
`thread_id`, por lo que la memoria de conversación queda aislada por sesión, igual que se
demostró en la sección 12 con `thread-A`/`thread-B`.


In [ ]:
print(
    "La demo interactiva en vivo se ejecuta como app independiente:\n\n"
    "    streamlit run streamlit_app.py\n\n"
    "Abre http://localhost:8501 en el navegador para chatear con el agente de forma visual."
)


La demo interactiva en vivo se ejecuta como app independiente:

    streamlit run streamlit_app.py

Abre http://localhost:8501 en el navegador para chatear con el agente de forma visual.


## 📝 14. Conclusiones

- Se construyó una base de conocimiento vectorial en ChromaDB con embeddings de Gemini a partir
  de 4 documentos oficiales de Micron Technology (2 informes 10-K + 2 proxy statements),
  superando el mínimo exigido de 3 documentos / ~20 páginas.
- El agente, implementado con `create_agent` de LangGraph/LangChain sobre Gemini, decide de
  forma autónoma cuándo consultar la base de conocimiento (RAG agéntico) y respeta las
  limitaciones definidas en el system prompt (no da asesoría de inversión, admite cuando no
  tiene información suficiente, cita sus fuentes).
- La memoria de conversación (`InMemorySaver` + `thread_id`) se verificó con preguntas de
  seguimiento que dependen explícitamente de respuestas anteriores dentro del mismo thread, y se
  confirmó el aislamiento entre threads distintos.
- **Extensión opcional (bonus) implementada:** interfaz de chat visual con **Streamlit**
  (`streamlit_app.py`), que reutiliza la misma base de conocimiento y el mismo agente RAG
  definidos en este notebook. Se lanza con `streamlit run streamlit_app.py`.
